In [0]:
#create the three schemas in unity catalog

from pyspark.sql import functions as F

for layer in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{layer}")

In [0]:
#bronze, raw as-is plus where each row came from

bronze = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/aemo_raw/")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.price_demand")
print("Bronze rows:", spark.table("workspace.bronze.price_demand").count())

In [0]:
# silver, cleaned and typed

silver = (
    spark.table("workspace.bronze.price_demand")
    .select(
        F.col("REGION").alias("region_code"),
        F.to_timestamp_ntz("SETTLEMENTDATE", F.lit("yyyy/MM/dd HH:mm:ss")).alias("interval_end_at"),
        F.col("RRP").cast("decimal(12,2)").alias("price_aud_per_mwh"),
        F.col("TOTALDEMAND").cast("decimal(12,2)").alias("demand_mw"),
        "_source_file",
        "_ingested_at",
    )
    .withColumn("interval_start_at", F.col("interval_end_at") - F.expr("INTERVAL 5 MINUTES"))
    .withColumn("interval_date", F.to_date("interval_start_at"))
    .withColumn("is_negative_price", F.col("price_aud_per_mwh") < 0)
    .filter(F.col("interval_end_at").isNotNull())
    .dropDuplicates(["region_code", "interval_end_at"])
)

silver.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.price_demand")
print("Silver rows:", spark.table("workspace.silver.price_demand").count())

In [0]:
# quality checks (the same idea as your dbt tests)

s = spark.table("workspace.silver.price_demand")

checks = {
    "rows (expect 44,640)": s.count(),
    "null timestamps (expect 0)": s.filter(F.col("interval_end_at").isNull()).count(),
    "null prices (expect 0)": s.filter(F.col("price_aud_per_mwh").isNull()).count(),
    "duplicate region-intervals (expect 0)": s.count() - s.dropDuplicates(["region_code", "interval_end_at"]).count(),
    "prices below -1000 floor (expect 0)": s.filter(F.col("price_aud_per_mwh") < -1000).count(),
}
for name, value in checks.items():
    print(f"{name}: {value:,}")

In [0]:
#Delta’s history

display(spark.sql("DESCRIBE HISTORY workspace.silver.price_demand"))